# 7장. 내 파일에 맞게 고치기: AI에게 맡기는 부분과 내가 확인하는 부분

새 지점(제주점) 파일은 열 이름이 영어다. 회사 데이터를 AI에 붙여 넣지 않고 도움을 받는 순서를 따라가 본다.
이 노트북은 AI를 호출하지 않는다. AI에게 보낼 글을 만들고, 받은 답을 검사하는 코드만 있다.

In [1]:
import random
import sys
from pathlib import Path

import pandas as pd

sys.path.insert(0, "..")
import weekly_report as wr

DATA = Path("../data")
WORK = Path("../work")
new_file = DATA / "extra" / "제주점_주간판매.xlsx"

## 1. 새 파일은 지금 코드로 읽히지 않는다

In [2]:
try:
    wr.read_branch_file(new_file)
except ValueError as error:
    print(error)

제주점_주간판매.xlsx: 열 이름이 적힌 줄을 찾지 못했습니다. 위쪽 10줄 안에 ['전표번호', '판매일', '품목코드', '품목명', '수량', '단가', '금액'] 같은 이름이 있어야 합니다.


## 2. 모양만 남긴 가짜 예시 만들기
AI에게는 열 이름과 값의 생김새만 보여 주면 된다. 글자는 같은 종류의 다른 글자로, 숫자는 자릿수만 같은 다른 숫자로 바꾼다.

In [3]:
def mask_text(text, rng):
    """글자 종류(숫자, 영문, 한글)와 길이는 두고 내용만 바꾼다."""
    out = []
    for ch in text:
        if ch.isdigit():
            out.append(str(rng.randint(0, 9)))
        elif "A" <= ch <= "Z":
            out.append(rng.choice("ABCDEFGHJKLMNPQRSTUVWXYZ"))
        elif "a" <= ch <= "z":
            out.append(rng.choice("abcdefghijkmnopqrstuvwxyz"))
        elif "가" <= ch <= "힣":
            out.append(rng.choice("가나다라마바사아자차카타파하"))
        else:
            out.append(ch)
    return "".join(out)


def fake_sample(path, rows=5, seed=0):
    """파일의 열 이름과 값의 생김새만 남긴, 내용은 지어낸 표를 만든다."""
    rng = random.Random(seed)
    real = pd.read_excel(path, nrows=50)
    fake = {}
    for col in real.columns:
        values = real[col].dropna()
        if pd.api.types.is_datetime64_any_dtype(values):
            fake[col] = [f"2000-01-{rng.randint(1, 28):02d}" for _ in range(rows)]
        elif pd.api.types.is_numeric_dtype(values):
            digits = len(str(int(values.abs().max())))
            fake[col] = [rng.randint(1, 10 ** digits - 1) for _ in range(rows)]
        else:
            texts = values.astype(str).tolist()
            fake[col] = [mask_text(rng.choice(texts), rng) for _ in range(rows)]
    return pd.DataFrame(fake)


sample = fake_sample(new_file)
sample

,slip_no,date,sku,item,qty,unit_price,amount
0,PB-4876-475,2000-01-23,H-287,파사바 93마다,79,43945,284204
1,GS-2421-948,2000-01-28,C-587,FBV 카마 아나나,15,24954,122825
2,EK-1157-815,2000-01-03,K-841,다가하 CY 86하,63,31856,739596
3,LV-3877-840,2000-01-07,L-839,마자 3G 96차,76,2125,231170
4,AC-6097-535,2000-01-19,U-471,아아카 51바,81,95878,390134


진짜 파일의 첫 줄과 나란히 놓고, 진짜 값이 하나도 안 남았는지 눈으로 확인한다.

In [4]:
pd.read_excel(new_file, nrows=2)

,slip_no,date,sku,item,qty,unit_price,amount
0,JJ-1102-001,2026-11-02,P-101,A4 복사용지 1박스,10,24500,245000
1,JJ-1102-002,2026-11-02,P-201,물티슈 10팩,3,11900,35700


## 3. AI에게 보낼 글 만들기
무엇을 주고, 무엇을 받고 싶고, 무엇을 하지 말아야 하는지를 한 번에 적는다. 아래 출력을 그대로 복사해 붙이면 된다.

In [5]:
def build_prompt(sample, standard, known):
    return f"""엑셀 파일을 파이썬 pandas로 읽어 주간 보고서를 만들고 있다.
아래는 새로 받은 파일의 열 이름과 예시 다섯 줄이다. 값은 생김새만 같고 내용은 지어낸 것이다.

{sample.to_csv(index=False).strip()}

내 코드가 쓰는 표준 열 이름: {standard}
이미 연결해 둔 이름: {known}

부탁: 새 파일의 열 이름을 표준 열 이름에 연결하는 파이썬 딕셔너리를 만들어 줘.

조건
1. {{"새 파일 열 이름": "표준 열 이름"}} 모양의 딕셔너리 하나만 답한다.
2. 맞는 열이 없는 표준 이름은 지어내지 말고, 딕셔너리 아래에 '없음: 이름'이라고 적는다.
3. 예시에 없는 열 이름은 쓰지 않는다.
4. 확실하지 않은 연결은 딕셔너리 아래에 한 줄로 이유를 적는다."""


prompt = build_prompt(sample, wr.STANDARD, wr.COLUMN_NAMES)
print(prompt)

엑셀 파일을 파이썬 pandas로 읽어 주간 보고서를 만들고 있다.
아래는 새로 받은 파일의 열 이름과 예시 다섯 줄이다. 값은 생김새만 같고 내용은 지어낸 것이다.

slip_no,date,sku,item,qty,unit_price,amount
PB-4876-475,2000-01-23,H-287,파사바 93마다,79,43945,284204
GS-2421-948,2000-01-28,C-587,FBV 카마 아나나,15,24954,122825
EK-1157-815,2000-01-03,K-841,다가하 CY 86하,63,31856,739596
LV-3877-840,2000-01-07,L-839,마자 3G 96차,76,2125,231170
AC-6097-535,2000-01-19,U-471,아아카 51바,81,95878,390134

내 코드가 쓰는 표준 열 이름: ['전표번호', '판매일', '품목코드', '품목명', '수량', '단가', '금액']
이미 연결해 둔 이름: {'일자': '판매일', '코드': '품목코드', '상품명': '품목명', '판매수량': '수량', '판매단가': '단가', '판매금액': '금액'}

부탁: 새 파일의 열 이름을 표준 열 이름에 연결하는 파이썬 딕셔너리를 만들어 줘.

조건
1. {"새 파일 열 이름": "표준 열 이름"} 모양의 딕셔너리 하나만 답한다.
2. 맞는 열이 없는 표준 이름은 지어내지 말고, 딕셔너리 아래에 '없음: 이름'이라고 적는다.
3. 예시에 없는 열 이름은 쓰지 않는다.
4. 확실하지 않은 연결은 딕셔너리 아래에 한 줄로 이유를 적는다.


## 4. 받은 답을 검사하기
AI는 없는 열 이름을 그럴듯하게 적을 때가 있다. 답을 코드에 넣기 전에 실제 파일과 견준다.
아래 `answer_wrong`은 그런 답을 흉내 내어 직접 적은 것이다(`unit_price`를 `price`로 적었다).

In [6]:
def check_mapping(answer, path, standard):
    """AI가 준 열 이름 연결이 실제 파일과 맞는지 본다. 문제를 글로 돌려준다."""
    real_columns = [str(c).strip() for c in pd.read_excel(path, nrows=0).columns]
    problems = []
    for source in answer:
        if source not in real_columns:
            problems.append(f"파일에 없는 열 이름: {source}")
    targets = list(answer.values())
    for name in standard:
        if targets.count(name) == 0:
            problems.append(f"연결되지 않은 표준 열: {name}")
        if targets.count(name) > 1:
            problems.append(f"두 번 연결된 표준 열: {name}")
    return problems


answer_wrong = {
    "slip_no": "전표번호", "date": "판매일", "sku": "품목코드", "item": "품목명",
    "qty": "수량", "price": "단가", "amount": "금액",
}
for problem in check_mapping(answer_wrong, new_file, wr.STANDARD):
    print("문제:", problem)

문제: 파일에 없는 열 이름: price


In [7]:
answer = {
    "slip_no": "전표번호", "date": "판매일", "sku": "품목코드", "item": "품목명",
    "qty": "수량", "unit_price": "단가", "amount": "금액",
}
print("문제:", check_mapping(answer, new_file, wr.STANDARD) or "없음")

문제: 없음


## 5. 고칠 곳은 한 군데
검사를 통과한 연결을 `COLUMN_NAMES`에 더하면 나머지 코드는 그대로 돈다.
여기서는 실행 중에만 더했다. 계속 쓰려면 `weekly_report.py` 위쪽의 `COLUMN_NAMES`에 같은 줄을 적어 넣는다.

In [8]:
wr.COLUMN_NAMES.update(answer)

jeju = wr.read_branch_file(new_file)
jeju_sales, jeju_check, jeju_notes = wr.clean(jeju, "2026-11-02")
print(
    f"제주점 {len(jeju_sales)}건, 합계 {jeju_sales['금액'].sum():,}원, "
    f"확인 필요 {len(jeju_check)}건"
)
jeju_sales.head(3)

제주점 20건, 합계 1,690,500원, 확인 필요 0건


,지점,원본파일,전표번호,판매일,품목코드,품목명,수량,단가,금액
0,제주점,제주점_주간판매.xlsx,JJ-1102-001,2026-11-02,P-101,A4 복사용지 1박스,10,24500,245000
1,제주점,제주점_주간판매.xlsx,JJ-1102-002,2026-11-02,P-201,물티슈 10팩,3,11900,35700
2,제주점,제주점_주간판매.xlsx,JJ-1102-003,2026-11-02,P-102,유성펜 12자루,5,6800,34000


## 6. 고친 뒤에는 예전 결과와 견준다
코드를 고쳤으면 원래 되던 것이 그대로 되는지 봐야 한다. 3장에서 저장해 둔 결과가 기준이다.

In [9]:
def compare_totals(before, after):
    """지점별 금액 합계를 고치기 전과 뒤로 나란히 놓는다."""
    table = pd.DataFrame({
        "고치기 전": before.groupby("지점")["금액"].sum(),
        "고친 뒤": after.groupby("지점")["금액"].sum(),
    }).fillna(0).astype("int64")
    table["차이"] = table["고친 뒤"] - table["고치기 전"]
    return table


baseline = pd.read_csv(WORK / "03_clean.csv", parse_dates=["판매일"])
after, _, _ = wr.clean(wr.read_week("2026-11-02"), "2026-11-02")
compare_totals(baseline, after)

,고치기 전,고친 뒤,차이
지점,,,
강남점,8004800,8004800,0
광주점,4858200,4858200,0
대전점,4060700,4060700,0
부산점,7020900,7020900,0
수원점,5975100,5975100,0


차이가 모두 0이면 기존 다섯 지점 결과는 그대로다.

## 실습. 그럴듯하지만 틀린 수정 잡아내기
"중복은 날짜, 품목, 수량이 같은 줄로 지우면 더 간단하다"는 제안을 받았다고 하자. 그대로 해 보고 표로 견준다.

In [10]:
suggested = baseline.drop_duplicates(subset=["지점", "판매일", "품목코드", "수량"])
print("사라진 줄:", len(baseline) - len(suggested))
compare_totals(baseline, suggested)

사라진 줄: 2


,고치기 전,고친 뒤,차이
지점,,,
강남점,8004800,8004800,0
광주점,4858200,4858200,0
대전점,4060700,4043000,-17700
부산점,7020900,7020900,0
수원점,5975100,5961500,-13600
